# Container Bay XML Loader

Pulls WardLayout XML from CAMIS (via the `CAM_LIVE_ODBC` linked server), parses the `Bay`/`Container` -> `Placement` structure, and loads the result into:

`[Production_CaMISStaging].[xml].[container_bay_xml]`

**Context:** This is "Solution C" from the corridor-care / bed occupancy work — classifying "no bay" bed episodes (chairs, containers) by their actual container type (e.g. `NEXT PATIENT` = true corridor care vs `Virtual Discharge Area` / `SWAP AREA` = not corridor care). `PlacementID` is the join key expected to match `BedIdentifierLOCAL` in `ips_bed` / `tbl_WardBed`.

Built on the same connection pattern as `Bed_occupancy_xml.ipynb` (SQLAlchemy + pyodbc, trusted Windows connection).

**Confirmed XML structure** (from a real ward A4 export):
```
<WardLayout objectId="A4">
  <Bay id="1" type="B">              <- real bay (B) or side room (R)
    <Placement id="11322" type="BED">
      <Caption><Content>Bed 6</Content></Caption>
    </Placement>
    <Caption><Content>Bay 2</Content></Caption>   <- Bay's own name
  </Bay>
  <Container>                         <- no id/type attributes
    <Placement id="21669" type="CHAIR">
      <Caption><Content>Chair A</Content></Caption>
    </Placement>
    <Caption><Content>NEXT PATIENT</Content></Caption>  <- Container's own name
  </Container>
  <Object>...</Object>                <- decorative text labels, NOT a bed location - ignored
</WardLayout>
```

Run the cells below **in order**. Each one prints output so you can sanity-check before moving on.

## 1. Imports

`pyodbc` + `sqlalchemy` talk to SQL Server, `pandas` holds the results, `xml.etree.ElementTree` parses the WardLayout XML.

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
import xml.etree.ElementTree as ET
from datetime import datetime

## 2. Configuration

The main cell you'll edit. `SOURCE_SERVER`/`TARGET_SERVER` are SQL Server host names — `CAM_LIVE_ODBC` is a **linked server** referenced inside the query below, not a host in its own right. `WARD_OBJECT_IDS` controls which ward(s) get pulled when auto-discovery is off.

`MIN_ROW_COUNT_RATIO` and `FORCE_OVERWRITE` guard the full-refresh strategy (discover all wards, replace the table each run): if a run's new row count is suspiciously smaller than what's already in the table, the write step refuses to overwrite rather than silently replacing good data with a partial/failed pull. Set `FORCE_OVERWRITE = True` only when a big drop is genuinely expected (e.g. wards were decommissioned).

In [ ]:
# Source: same server your Bed_occupancy_xml.ipynb connects to.
# CAM_LIVE_ODBC is a LINKED SERVER underneath this instance, not a
# standalone host - that's why the SQL uses 4-part naming.
SOURCE_SERVER = "INFODW1"          # <-- confirm this matches your environment

# Destination: where the parsed table gets written
TARGET_SERVER = "INFODW1"          # <-- update if staging lives on a different server
TARGET_DATABASE = "Production_CaMISStaging"
TARGET_SCHEMA = "xml"
TARGET_TABLE = "container_bay_xml"

# The ward(s) you want to pull. Extend this list to run multiple wards
# through the same query/parse/load pipeline in one go.
WARD_OBJECT_IDS = ["5"]

# IMPORTANT DISCOVERY: the error from testing OPENQUERY revealed that
# CAM_LIVE_ODBC is backed by INFORMIX, not SQL Server - the remote engine
# rejected "NVARCHAR(MAX)" because that's SQL-Server-only type syntax.
# This also means xdi_xml's true type lives in Informix's type system,
# which has a well-known practical VARCHAR/NVARCHAR/LVARCHAR ceiling around
# 32739 characters - suspiciously close to the ~32543-char truncation point
# we're seeing. That raises a real possibility this ISN'T a transport-layer
# truncation fixable by any query technique at all, but a genuine Informix
# column-size limit hit when this document was originally SAVED by CaMIS.
#
# Default is False (4-part naming) because it's the only option confirmed
# not to crash. Set to True to try the experimental Informix-syntax
# OPENQUERY rewrite below - it may fix a syntax problem without fixing the
# underlying truncation, if the Informix-column-limit theory is correct.
USE_OPENQUERY = False
# Data-loss guard (Option A full-refresh safety net): before overwriting
# the table, compare the new row count against what's already there.
# If the new pull is suspiciously smaller, refuse to overwrite rather
# than silently replacing good data with a partial/failed run.
MIN_ROW_COUNT_RATIO = 0.7   # new count must be at least 70% of existing
FORCE_OVERWRITE = False     # set True to bypass the guard (e.g. wards
                            # were genuinely decommissioned)


## 3. Connection Helper

Two things happen here, both fixes for real errors hit earlier:

- **`get_installed_sql_driver()`** auto-detects which SQL Server ODBC driver is actually installed on this machine, rather than hardcoding a version — hardcoding `"ODBC Driver 17 for SQL Server"` failed with an `IM002` error on a machine that didn't have that exact driver registered.
- **`get_engine()`** builds the connection string using the `odbc_connect` format instead of SQLAlchemy's `mssql+pyodbc://@server?driver=...` URL style. That URL style has a real bug: if a host is given with **no** database, SQLAlchemy's dialect silently treats it as a named DSN lookup (`dsn=INFODW1`) instead of a driver connection — which is exactly what caused a second, identical-looking `IM002` error. `odbc_connect` builds the raw ODBC string directly and sidesteps that bug entirely.

In [ ]:
import re
import urllib.parse

def get_installed_sql_driver():
    """
    Auto-detect the installed SQL Server ODBC driver name. Hardcoding
    'ODBC Driver 17 for SQL Server' fails with IM002 ("Data source name
    not found") if your machine actually has a different version (18,
    13.1, SQL Server Native Client, etc.) or a differently spelled name.

    Prefers the highest-numbered 'ODBC Driver NN for SQL Server' if one
    is installed; otherwise falls back to any other SQL Server driver
    found (e.g. legacy 'SQL Server Native Client').
    """
    drivers = [d for d in pyodbc.drivers() if "SQL Server" in d]
    if not drivers:
        raise RuntimeError(
            "No SQL Server ODBC driver found on this machine. "
            "Install 'ODBC Driver 17 for SQL Server' (or 18) from Microsoft, "
            "or ask Simon Newey to install it."
        )

    def odbc_version(name):
        m = re.match(r"ODBC Driver (\d+) for SQL Server", name)
        return int(m.group(1)) if m else -1

    odbc_drivers = [d for d in drivers if odbc_version(d) >= 0]
    chosen = max(odbc_drivers, key=odbc_version) if odbc_drivers else drivers[0]

    print(f"Detected ODBC drivers: {drivers}")
    print(f"Using: {chosen}")
    return chosen


SQL_DRIVER = get_installed_sql_driver()


def get_engine(server, database=None):
    """
    Create a SQLAlchemy engine using Windows trusted connection.

    Uses the 'odbc_connect' style rather than
    'mssql+pyodbc://@server?driver=...' - the latter relies on SQLAlchemy's
    URL query-string parsing to correctly reconstruct a driver name that
    contains spaces, which has been unreliable on this environment (IM002
    errors even with a correct, installed driver). Building the raw ODBC
    connection string directly - the same format proven to work in
    test_sql_simple.py - and passing it via odbc_connect avoids that
    parsing step entirely.
    """
    parts = [f"DRIVER={{{SQL_DRIVER}}}", f"SERVER={server}"]
    if database:
        parts.append(f"DATABASE={database}")
    parts.append("Trusted_Connection=yes")
    raw_conn_str = ";".join(parts) + ";"

    quoted = urllib.parse.quote_plus(raw_conn_str)
    conn_str = f"mssql+pyodbc:///?odbc_connect={quoted}"
    return create_engine(conn_str)

### 3b. Diagnostic: raw pyodbc connection test

Bypasses SQLAlchemy entirely and connects the same way `test_sql_simple.py` did. If **this** cell fails, the problem is the driver, server, or network — not this notebook's SQLAlchemy wiring. If it succeeds but a later cell fails, that narrows the problem to SQLAlchemy specifically.

In [ ]:
_test_driver = SQL_DRIVER
_test_conn_str = f"Driver={{{_test_driver}}};Server={SOURCE_SERVER};Trusted_Connection=yes;"
print(f"Testing raw pyodbc connection: {_test_conn_str}")

try:
    _test_conn = pyodbc.connect(_test_conn_str, timeout=10)
    print("✅ Raw pyodbc connection succeeded")
    _test_conn.close()
except Exception as e:
    print(f"❌ Raw pyodbc connection FAILED: {e}")
    print("\nThis means the problem is the driver/server/network itself,")
    print("not the notebook's SQLAlchemy wiring. Next steps:")
    print("  1. Confirm SOURCE_SERVER is reachable (ping/VPN/network)")
    print("  2. Confirm your account has SQL Server login access")
    print("  3. Try the exact server name your DBA/Simon Newey gave you")
    raise

### Auto-Discover Ward Codes

`explain_sql_error()` (clean error summaries instead of raw tracebacks) is defined here, since this is now the earliest cell that needs it.

Set `AUTO_DISCOVER_ALL_WARDS = True` to pull every ward code that has a WardLayout document straight from CaMIS and load them all in this run - no need to type/maintain the list by hand. Set it to `False` to use the manually-typed `WARD_OBJECT_IDS` list from the config cell above instead.

This discovery query only returns ward codes (tiny strings) - never the XML itself - so it's fast and carries none of the large-document transit-corruption risk we saw earlier.

In [ ]:
def explain_sql_error(e, query_used):
    """
    Print a short, readable summary of a SQL failure instead of the
    ~40-line nested SQLAlchemy/pandas traceback. The genuinely useful
    part of a pyodbc/SQLAlchemy error is almost always in the first
    few hundred characters (the driver's own error message) - the rest
    is internal call-stack noise.
    """
    error_text = str(e)
    print(f"\n{'='*70}")
    print(f"❌ QUERY FAILED")
    print(f"{'='*70}")
    print(f"\nDriver/database message:")
    print(f"  {error_text[:500]}")
    print(f"\nQuery attempted (first 300 chars):")
    print(f"  {' '.join(query_used.split())[:300]}")

    lowered = error_text.lower()
    print(f"\nLikely cause:")
    if "syntax error" in lowered and "informix" in lowered:
        print("  Informix rejected the inner OPENQUERY SQL as invalid syntax.")
        print("  This confirms CAM_LIVE_ODBC is Informix-backed, not SQL Server -")
        print("  T-SQL-specific syntax (e.g. NVARCHAR(MAX)) will not work inside")
        print("  an OPENQUERY string sent to it. Try USE_OPENQUERY = False, or")
        print("  adjust build_query_openquery()'s inner SQL to valid Informix syntax.")
    elif "login failed" in lowered or "28000" in error_text:
        print("  Authentication/permission issue - check your account has access")
        print("  to this server/database, or ask Simon Newey to grant it.")
    elif "timeout" in lowered:
        print("  Connection or query timed out - check network/VPN, or the query")
        print("  may need a longer timeout for this data volume.")
    elif "invalid object name" in lowered or "invalid column name" in lowered:
        print("  A table/column name in the query doesn't exist as written - check")
        print("  spelling and that the linked server/database/schema names match.")
    else:
        print("  Not one of the previously-seen failure patterns - read the driver")
        print("  message above for specifics, or share it for further diagnosis.")
    print(f"{'='*70}\n")


AUTO_DISCOVER_ALL_WARDS = True   # False = use the manual WARD_OBJECT_IDS list above instead

source_engine = get_engine(SOURCE_SERVER)

if AUTO_DISCOVER_ALL_WARDS:
    discover_query = """
    SELECT DISTINCT s2.xiv_value AS WardCode
    FROM      [CAM_LIVE_ODBC].[CAMIS].[camis].xss_docinstance di,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_document d,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_idxvalstring s2,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_indexdef i2
    WHERE
        di.xdc_docid = d.xdc_docid
        AND s2.xdi_instance = di.xdi_instance
        AND s2.xid_indexid = i2.xid_indexid
        AND i2.xid_indexname = 'ObjectId'
        AND di.xdi_replacetime IS NULL
        AND d.xdt_doctypeid = (
            SELECT xdt_doctypeid
            FROM [CAM_LIVE_ODBC].[CAMIS].[camis].xss_documenttype
            WHERE xdt_doctypename = 'WardLayout'
        )
    ORDER BY WardCode
    """
    print("Auto-discovering all ward codes with a WardLayout document...")
    try:
        with source_engine.connect() as discover_conn:
            df_wards = pd.read_sql_query(discover_query, discover_conn)
        WARD_OBJECT_IDS = df_wards["WardCode"].tolist()
        print(f"✅ Discovered {len(WARD_OBJECT_IDS)} ward(s)")
        print(WARD_OBJECT_IDS)
    except Exception as e:
        explain_sql_error(e, discover_query)
        raise RuntimeError("Ward discovery failed - see above") from None
else:
    print(f"Using manually configured WARD_OBJECT_IDS: {WARD_OBJECT_IDS}")

## 4. Run the WardLayout Query

**Major finding from testing OPENQUERY:** the error revealed `CAM_LIVE_ODBC` is backed by **Informix**, not SQL Server - it rejected `NVARCHAR(MAX)` as a syntax error, because that's SQL-Server-only type syntax. This also means `xdi_xml`'s true type lives in Informix's own type system, which has a well-documented practical ceiling of ~32,739 characters for `VARCHAR`/`NVARCHAR`/`LVARCHAR`-family columns - suspiciously close to the ~32,543-character truncation point observed earlier.

**This changes the diagnosis.** It's now a real possibility that the truncation isn't happening in transit at all - it may be a genuine Informix column-size limit that was hit when this document was originally *saved* by CaMIS, long before this notebook ever queried it. If so, no query technique (OPENQUERY included) can retrieve more than what's actually stored.

**What's in this cell now:**
- `USE_OPENQUERY` defaults to `False` - the 4-part-naming query is the only form confirmed not to crash
- `build_query_openquery()` has been fixed to use valid Informix syntax (no more `NVARCHAR(MAX)`) as a best-effort, **unverified** attempt if you want to keep testing it
- The self-check after fetching now flags explicitly when a truncated length is close to Informix's known ceiling, and suggests the two concrete next tests: try a smaller ward (cheap, no code change), or ask Simon Newey/DBA to check `xdi_xml`'s declared length directly in Informix's own system catalog (the only way to get a fully definitive answer)

In [ ]:
def build_query_4part(ward_object_ids):
    """Original 4-part linked-server naming. Confirmed via diagnostics
    to return xdi_xml truncated at ~32.5k characters - but it's the
    only query form confirmed NOT to crash, so it's the safe default
    while the truncation's true cause (transport vs genuine source-side
    Informix column limit) is still unresolved."""
    ids_list = ", ".join(f"'{w}'" for w in ward_object_ids)
    return f"""
    SELECT CAST(di.xdi_xml AS NVARCHAR(MAX)) AS xdi_xml, s2.xiv_value
    FROM      [CAM_LIVE_ODBC].[CAMIS].[camis].xss_docinstance di,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_document d,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_idxvalstring s2,
              [CAM_LIVE_ODBC].[CAMIS].[camis].xss_indexdef i2
    WHERE
        di.xdc_docid = d.xdc_docid
        AND s2.xdi_instance = di.xdi_instance
        AND s2.xid_indexid = i2.xid_indexid
        AND i2.xid_indexname = 'ObjectId'
        AND di.xdi_replacetime IS NULL
        AND d.xdt_doctypeid = (
            SELECT xdt_doctypeid
            FROM [CAM_LIVE_ODBC].[CAMIS].[camis].xss_documenttype
            WHERE xdt_doctypename = 'WardLayout'
        )
        AND s2.xiv_value IN ({ids_list})
    """


def build_query_openquery(ward_object_ids):
    """
    EXPERIMENTAL / UNVERIFIED - the remote system behind CAM_LIVE_ODBC
    is Informix, not SQL Server (confirmed by an earlier syntax error).
    This version removes the SQL-Server-specific CAST(...AS NVARCHAR(MAX))
    and lets Informix return xdi_xml in its own native type - a best-
    effort fix for the SYNTAX problem only.

    This does NOT necessarily fix the truncation itself. Informix's
    VARCHAR/NVARCHAR/LVARCHAR types have a well-known practical ceiling
    around 32739 characters - close enough to the ~32543 we're seeing
    that the truncation may be a genuine Informix column-size limit hit
    when this document was originally saved, not something happening in
    transit at all.
    """
    ids_list = ", ".join(f"''{w}''" for w in ward_object_ids)
    inner_query = f"""
        SELECT di.xdi_xml, s2.xiv_value
        FROM      camis.xss_docinstance di,
                  camis.xss_document d,
                  camis.xss_idxvalstring s2,
                  camis.xss_indexdef i2
        WHERE
            di.xdc_docid = d.xdc_docid
            AND s2.xdi_instance = di.xdi_instance
            AND s2.xid_indexid = i2.xid_indexid
            AND i2.xid_indexname = ''ObjectId''
            AND di.xdi_replacetime IS NULL
            AND d.xdt_doctypeid = (
                SELECT xdt_doctypeid
                FROM camis.xss_documenttype
                WHERE xdt_doctypename = ''WardLayout''
            )
            AND s2.xiv_value IN ({ids_list})
    """
    inner_query_clean = " ".join(inner_query.split())
    return f"SELECT * FROM OPENQUERY(CAM_LIVE_ODBC, '{inner_query_clean}')"


def build_query(ward_object_ids):
    return build_query_openquery(ward_object_ids) if USE_OPENQUERY else build_query_4part(ward_object_ids)


# source_engine was already created in the discovery cell above
query = build_query(WARD_OBJECT_IDS)
print(f"Running WardLayout query for ward(s): {WARD_OBJECT_IDS} "
      f"(via {'OPENQUERY (experimental)' if USE_OPENQUERY else '4-part naming'})...")

df_raw = None
try:
    # Run SET TEXTSIZE and the query on the SAME connection - see cell
    # above for why that matters.
    with source_engine.connect() as source_conn:
        source_conn.execute(text("SET TEXTSIZE 2147483647"))
        df_raw = pd.read_sql_query(query, source_conn)
    print(f"✅ Retrieved {len(df_raw)} row(s)")
except Exception as e:
    explain_sql_error(e, query)
    raise RuntimeError("Query failed - see diagnostic above for details") from None

df_raw.head()

### Shared error-handling helpers

Two functions used by both the completeness check below and the real parser further down:

- **`diagnose_parse_error()`** - prints full, non-truncated context for a parse failure (exact line/column, surrounding text, whether the doc closes properly) - not squeezed into a table row where it gets cut off.
- **`sanitize_xml()`** - fixes two independent, confirmed corruption patterns before giving up: (1) bare `&` characters not part of a valid entity - fixes free-text captions like `"A & E"`; (2) NUL bytes and spurious newlines - confirmed via a real example (ward ERU) where a large value fetched through the CAM_LIVE_ODBC linked server came back with a stray `\x00` byte and extra line breaks spliced into otherwise-correct content, a classic buffer-boundary artifact in ODBC chunk reassembly. Both fixes are only ever applied as a fallback when raw parsing fails - documents that already parse cleanly are never touched.

In [ ]:
import re

def diagnose_parse_error(xml_string, ward_code, error):
    """
    Print the exact context around an XML parse failure - the raw error
    message alone ('not well-formed (invalid token): line N, column M')
    doesn't say what's actually wrong. This shows the offending line,
    whether the document is properly closed, and whether there's more
    than one XML declaration (a sign of concatenated/corrupted content).
    """
    print(f"\n{'='*70}")
    print(f"XML PARSE DIAGNOSTIC — ward {ward_code}")
    print(f"{'='*70}")
    print(f"Error: {error}")
    print(f"String length: {len(xml_string)} characters")
    print(f"Starts with BOM: {xml_string.startswith(chr(0xFEFF))}")
    print(f"Number of '<?xml' declarations found: {xml_string.count('<?xml')}")
    print(f"Ends with '</WardLayout>': {xml_string.rstrip().endswith('</WardLayout>')}")
    lines = xml_string.splitlines()
    print(f"Line count (split on \\n): {len(lines)}")

    if hasattr(error, "position"):
        err_line, err_col = error.position
        print(f"\nContext around line {err_line}, column {err_col}:")
        start = max(1, err_line - 1)
        end = min(len(lines), err_line + 1)
        for ln in range(start, end + 1):
            marker = ">>> " if ln == err_line else "    "
            content = lines[ln - 1] if ln - 1 < len(lines) else "(no such line)"
            print(f"{marker}Line {ln}: {content[:150]}")
            if ln == err_line and err_col < len(content):
                # point directly at the offending character
                pointer = " " * (len(marker) + len(f"Line {ln}: ") + min(err_col, 150)) + "^"
                print(pointer)
                print(f"    Character at that position: {content[err_col:err_col+1]!r}")
    print(f"{'='*70}\n")


def sanitize_xml(xml_string):
    """
    Two independent, safe fixes for known corruption patterns:

    1. Escape bare '&' characters not already part of a valid XML
       entity reference (&amp; &lt; &gt; &apos; &quot; &#123; &#xAB;) -
       fixes free-text captions like "A & E". Never touches already-
       valid entities.

    2. Strip NUL bytes (\x00) and newline/carriage-return characters.
       Confirmed via a real example (ward ERU): large XML values
       fetched through the CAM_LIVE_ODBC linked server can come back
       with a stray NUL byte and extra newlines spliced in mid-content
       - a classic buffer-boundary artifact in ODBC chunk reassembly,
       most likely at the MSDASQL/Informix driver bridge. A NUL byte is
       NEVER valid in XML, so removing it is always safe. Stripping
       newlines is a slightly stronger assumption - but every real
       WardLayout document we've inspected (A4, ward 5, ERU's clean
       SSMS-copied reference) is legitimately single-line, so a
       newline appearing in a document that FAILS to parse is almost
       certainly a transit artifact, not real content. This is only
       ever applied as a recovery fallback - documents that already
       parse successfully as received are never touched.
    """
    fixed = re.sub(r'&(?!amp;|lt;|gt;|apos;|quot;|#\d+;|#x[0-9a-fA-F]+;)', '&amp;', xml_string)
    fixed = fixed.replace('\x00', '').replace('\r', '').replace('\n', '')
    return fixed


def try_parse_with_recovery(xml_string):
    """
    Try parsing as-is; if that fails, try again after sanitizing bare
    ampersands. Returns (root_or_None, used_sanitized, error_or_None).
    """
    try:
        return ET.fromstring(xml_string), False, None
    except ET.ParseError as first_error:
        sanitized = sanitize_xml(xml_string)
        if sanitized != xml_string:
            try:
                return ET.fromstring(sanitized), True, None
            except ET.ParseError:
                pass
        return None, False, first_error

## 4b. Document Completeness Check (this is your cheap multi-ward test)

Checks **every** row returned - not just the first - for whether it closes properly, and flags lengths close to Informix's known ~32,739-character ceiling.

**To test whether the truncation is ward-specific or systemic:** add more ward codes to `WARD_OBJECT_IDS` in the config cell (e.g. `["A4", "B2", "C1"]`), re-run from there, and this cell will show you a length/status line for every ward in one pass - no other code changes needed.

In [ ]:
INFORMIX_LIKELY_LIMIT_CHARS = 32739

print(f"{'Ward':<10} {'Length':>10}  {'Valid XML':<12}  {'Bays':>5}  {'Containers':>10}  {'Loose':>6}  Note")
print("-" * 90)

any_invalid = False
invalid_details = []  # (ward_code, xml_string, error) for full diagnostics after the table

for _, row in df_raw.iterrows():
    ward_code = row["xiv_value"]
    xml_string = row["xdi_xml"]
    length = len(xml_string)

    root, used_sanitized, error = try_parse_with_recovery(xml_string)

    note = ""
    bay_count = container_count = loose_count = "-"
    if root is not None:
        valid_label = "⚠️  fixed" if used_sanitized else "✅ yes"
        bay_count = len(root.findall("Bay"))
        container_count = len(root.findall("Container"))
        loose_count = len(root.findall("Placement"))
        notes = []
        if used_sanitized:
            notes.append("had invalid content (unescaped '&', NUL bytes, and/or "
                          "spurious newlines) - auto-fixed, parsed OK")
        if loose_count > 0:
            notes.append(f"{loose_count} placement(s) with NO Bay/Container wrapper")
        if bay_count == 0 and container_count == 0 and loose_count == 0:
            notes.append("no bays/containers/placements defined (legitimately empty ward)")
        note = "; ".join(notes)
    else:
        any_invalid = True
        valid_label = "❌ no"
        if abs(INFORMIX_LIKELY_LIMIT_CHARS - length) < 1000:
            note = "near Informix ~32739-char limit - likely source-side truncation"
        else:
            note = "parse error - see full diagnostic below table"
            invalid_details.append((ward_code, xml_string, error))

    print(f"{ward_code:<10} {length:>10}  {valid_label:<12}  {bay_count!s:>5}  {container_count!s:>10}  {loose_count!s:>6}  {note}")

print("-" * 90)

# Full, non-truncated diagnostics for anything that still failed even
# after the ampersand-sanitization fallback - printed separately below
# the table rather than squeezed into one row, so nothing gets cut off.
for ward_code, xml_string, error in invalid_details:
    diagnose_parse_error(xml_string, ward_code, error)

if not any_invalid:
    print("✅ All documents are valid (or auto-recoverable) XML.")
elif len(df_raw) > 1:
    print("\nIf only SOME wards are invalid: likely size-related (Informix limit)")
    print("or a data-quality issue specific to that ward's content.")
else:
    print("\nOnly one ward tested. Add more ward codes to WARD_OBJECT_IDS")
    print("above and re-run to compare.")

## 5. (Optional) Preview the Raw XML

Sanity check only — confirms the value that came back actually starts with a real `<?xml ...?>` declaration and looks intact, before trusting the parser in the next section.

In [ ]:
print(df_raw.iloc[0]["xdi_xml"][:1000])

## 6. XML Parsing Helpers

Two functions, run against the confirmed real structure of a ward export (see the structure diagram at the top of this notebook).

**`get_caption_text()`** — pulls the display name out of an element's own `Caption/Content` child (e.g. a Bay's `"Bay 2"`, a Container's `"NEXT PATIENT"`). Uses a direct-child path so it never accidentally grabs a nested Placement's caption instead.

In [ ]:
def get_caption_text(element):
    """
    Pull the display text out of an element's OWN Caption/Content child
    (e.g. Bay's "Bay 2", Container's "NEXT PATIENT", Placement's "Bed 6").
    Uses a direct-child path so it never picks up a nested Placement's
    caption by mistake.
    """
    content_el = element.find("Caption/Content")
    if content_el is not None and content_el.text:
        return content_el.text.strip()
    return None

**`parse_ward_layout_xml()`** — the main parser. Walks every `<Bay>` and `<Container>` under the root, and for each one, every direct-child `<Placement>` inside it, producing one row per placement with the parent's identity (bay number/type or container name) alongside the placement's own id/type/caption. `<Object>` elements (decorative text boxes like the ward title banner) are deliberately excluded. If the raw XML fails to parse (e.g. a caption with an unescaped `&`), it automatically retries via `sanitize_xml()` before giving up.

In [ ]:
def parse_ward_layout_xml(xml_string, ward_code_from_query):
    """
    Extract one row per Placement, grouped under its parent Bay or Container.

    Columns produced:
        WardCode          - ward object id (from the WardLayout root, cross-
                             checked against the value the query matched on)
        ParentElementType - 'Bay' or 'Container'
        ParentID          - Bay's id attribute (None for Container)
        ParentTypeCode    - Bay's type attribute: 'B' (bay) or 'R' (side room);
                             None for Container
        ParentCaption     - display name, e.g. 'Bay 2', 'Side Room A',
                             'NEXT PATIENT', 'Virtual Discharge Area', 'SWAP AREA'
        PlacementID       - the id CaMIS assigns to the bed/chair/trolley -
                             this is the join key expected to match
                             BedIdentifierLOCAL in ips_bed / tbl_WardBed
        PlacementType     - 'BED', 'CHAIR', 'TROLLEY' etc.
        PlacementCaption  - display name, e.g. 'Bed 6', 'Chair A', 'Trolley 1'

    <Object> elements (decorative text boxes like "Nurses Station" or the
    ward title banner) are deliberately NOT included - they aren't bed
    locations.
    """
    root, used_sanitized, parse_error = try_parse_with_recovery(xml_string)
    if root is None:
        raise parse_error
    if used_sanitized:
        print(f"⚠️  Ward {ward_code_from_query}: had invalid content (unescaped '&', "
              f"NUL bytes, and/or spurious newlines) - auto-fixed before parsing")
    ward_code = root.attrib.get("objectId", ward_code_from_query)

    if ward_code != ward_code_from_query:
        print(f"⚠️  Note: XML objectId '{ward_code}' differs from query match "
              f"value '{ward_code_from_query}' - using XML's objectId.")

    rows = []
    parent_elements = root.findall("Bay") + root.findall("Container")

    # Beds/chairs sometimes sit as DIRECT children of the WardLayout root -
    # no <Bay> or <Container> wrapper at all, just a decorative <Object>
    # text label nearby (e.g. "BAY 1 NORTH") that has no structural link to
    # the beds. Confirmed on a real ward (ERU): 16 beds this way, which the
    # original Bay/Container-only logic silently missed with no error.
    for placement_el in root.findall("Placement"):
        rows.append({
            "WardCode": ward_code,
            "ParentElementType": "Unassigned",
            "ParentID": None,
            "ParentTypeCode": None,
            "ParentCaption": None,
            "PlacementID": placement_el.attrib.get("id"),
            "PlacementType": placement_el.attrib.get("type"),
            "PlacementCaption": get_caption_text(placement_el),
        })

    for parent_el in parent_elements:
        parent_tag = parent_el.tag
        parent_id = parent_el.attrib.get("id")
        parent_type = parent_el.attrib.get("type")
        parent_caption = get_caption_text(parent_el)

        placements = parent_el.findall("Placement")

        if not placements:
            rows.append({
                "WardCode": ward_code,
                "ParentElementType": parent_tag,
                "ParentID": parent_id,
                "ParentTypeCode": parent_type,
                "ParentCaption": parent_caption,
                "PlacementID": None,
                "PlacementType": None,
                "PlacementCaption": None,
            })
            continue

        for placement_el in placements:
            rows.append({
                "WardCode": ward_code,
                "ParentElementType": parent_tag,
                "ParentID": parent_id,
                "ParentTypeCode": parent_type,
                "ParentCaption": parent_caption,
                "PlacementID": placement_el.attrib.get("id"),
                "PlacementType": placement_el.attrib.get("type"),
                "PlacementCaption": get_caption_text(placement_el),
            })

    return rows, used_sanitized

## 7. Parse Every Row Returned by the Query

Runs the parser (with the same ampersand/NUL/newline recovery fallback as the completeness check above) on every row from the query and combines the results into `df_parsed`. If a document still fails even after recovery, `diagnose_parse_error()` (defined earlier) prints full context so the failure is diagnosable, not just reported.

Two things added here for ongoing maintenance rather than one-off runs:

- **Data-quality columns baked into the output itself** — `DocumentLength`, `IsDocumentComplete`, `LoadTimestamp` get added to every row. Anyone querying `container_bay_xml` later can see whether a given ward's data needed auto-fixing and when it was last loaded, without needing to re-run this notebook to find out.
- **A run summary that makes partial failures impossible to miss** — a per-ward table (Success / Recovered / Empty / Failed) printed at the end, plus an explicit list of any failed wards, rather than a status buried mid-scroll in the per-ward log lines.

In [ ]:
load_timestamp = datetime.now()

all_rows = []
ward_status = []  # per-ward outcome, used for the run summary below

for _, row in df_raw.iterrows():
    ward_code = row["xiv_value"]
    xml_string = row["xdi_xml"]
    doc_length = len(xml_string)
    try:
        rows, used_sanitized = parse_ward_layout_xml(xml_string, ward_code)

        # Bake data-quality info directly into the output table, not just
        # into notebook output that disappears after the session ends -
        # anyone querying container_bay_xml later can see whether a given
        # ward's data needed auto-fixing, and when it was loaded.
        for r in rows:
            r["DocumentLength"] = doc_length
            r["IsDocumentComplete"] = not used_sanitized
            r["LoadTimestamp"] = load_timestamp

        all_rows.extend(rows)
        bay_count = sum(1 for r in rows if r["ParentElementType"] == "Bay")
        container_count = sum(1 for r in rows if r["ParentElementType"] == "Container")
        print(f"✓ Ward {ward_code}: {len(rows)} placement row(s) "
              f"({bay_count} in bays, {container_count} in containers)")

        status = "Recovered" if used_sanitized else ("Empty" if len(rows) == 0 else "Success")
        ward_status.append({"WardCode": ward_code, "Status": status,
                             "RowCount": len(rows), "DocLength": doc_length})
    except ET.ParseError as e:
        print(f"❌ Failed to parse XML for ward {ward_code}: {e}")
        diagnose_parse_error(xml_string, ward_code, e)
        ward_status.append({"WardCode": ward_code, "Status": "Failed",
                             "RowCount": 0, "DocLength": doc_length})

df_parsed = pd.DataFrame(all_rows)
print(f"\n✅ Parsed {len(df_parsed)} total row(s) across {len(df_raw)} ward document(s)")

# --- Run summary: a partial failure should never be easy to miss ---
df_status = pd.DataFrame(ward_status)
print(f"\n{'='*70}")
print("RUN SUMMARY")
print(f"{'='*70}")
print(df_status.to_string(index=False))

n_success = (df_status["Status"] == "Success").sum()
n_recovered = (df_status["Status"] == "Recovered").sum()
n_empty = (df_status["Status"] == "Empty").sum()
n_failed = (df_status["Status"] == "Failed").sum()
print(f"\n{n_success} succeeded cleanly, {n_recovered} recovered (auto-fixed), "
      f"{n_empty} legitimately empty, {n_failed} FAILED")
if n_failed > 0:
    failed_wards = df_status.loc[df_status["Status"] == "Failed", "WardCode"].tolist()
    print(f"⚠️  Failed ward(s): {failed_wards} - see diagnostics above for each")
print(f"{'='*70}")

df_parsed.head(15)

## 8. Breakdown by Container/Bay Name

**Stop and look here before continuing to the load.** This is where you confirm the container names for this ward match what's expected (e.g. `NEXT PATIENT` = true corridor care vs `SWAP AREA` / `Virtual Discharge Area` = not corridor care) before anything gets written to `Production_CaMISStaging`.

In [ ]:
if df_parsed.empty:
    print("No bays/containers/placements found for any requested ward -")
    print("nothing to break down. This is expected if the ward(s) genuinely")
    print("have none defined in CaMIS (see the §4b check above). If you")
    print("expected data here, double check WARD_OBJECT_IDS.")
    summary = pd.DataFrame(columns=["ParentElementType", "ParentCaption", "PlacementCount"])
else:
    summary = (
        df_parsed.groupby(["ParentElementType", "ParentCaption"])
        .size()
        .reset_index(name="PlacementCount")
    )
summary

## 9. Load Into SQL Server

Creates the `xml` schema in `Production_CaMISStaging` if it doesn't already exist — `to_sql()` will fail if the schema is missing.

In [ ]:
def ensure_schema_exists(engine, schema_name):
    """Create the target schema if it doesn't already exist."""
    try:
        with engine.connect() as conn:
            conn.execute(text(f"""
                IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = '{schema_name}')
                EXEC('CREATE SCHEMA {schema_name}')
            """))
            conn.commit()
        print(f"✓ Schema '{schema_name}' confirmed")
    except Exception as e:
        print(f"\n❌ Failed to create/confirm schema '{schema_name}'")
        print(f"  {str(e)[:400]}")
        print("  Likely cause: your account may not have CREATE SCHEMA permission")
        print(f"  on {TARGET_DATABASE} - ask Simon Newey to grant it, or to create")
        print(f"  the '{schema_name}' schema manually.")
        raise RuntimeError("Schema check/creation failed - see above") from None

target_engine = get_engine(TARGET_SERVER, TARGET_DATABASE)
ensure_schema_exists(target_engine, TARGET_SCHEMA)

## 10. Write the Parsed Table

This implements **Option A (full refresh)** as the ongoing maintenance strategy: rediscover all wards, reparse everything, and replace the table each run. Deletions and renames are handled for free this way — a ward that no longer exists in CaMIS simply isn't in the new pull, so it naturally drops out.

The risk with `if_exists="replace"` run unattended is a bad or partial pull silently wiping out previously-good data — that's what the row-count guard above (§2 config: `MIN_ROW_COUNT_RATIO`) protects against. If the new pull looks suspiciously small compared to what's already there, the write is skipped rather than applied.

In [ ]:
write_skipped = df_parsed.empty
skip_reason = "df_parsed was empty (no bays/containers/placements found)"

if write_skipped:
    print("⚠️  Nothing to write - df_parsed is empty (no bays/containers/")
    print("placements found for the requested ward(s)). Skipping the write")
    print("rather than attempting to create a table with no columns.")
    print("If this is unexpected, check §4b above and WARD_OBJECT_IDS.")
else:
    new_row_count = len(df_parsed)

    # --- Data-loss guard ---
    # A full-refresh strategy (discover all wards + if_exists="replace")
    # is only safe to run unattended if a bad or partial run can never
    # silently wipe out previously-good data. Compare against what's
    # already in the table before overwriting it.
    existing_row_count = None
    try:
        with target_engine.connect() as check_conn:
            result = check_conn.execute(text(
                f"SELECT COUNT(*) FROM [{TARGET_SCHEMA}].[{TARGET_TABLE}]"
            ))
            existing_row_count = result.scalar()
    except Exception:
        pass  # table doesn't exist yet - first run, nothing to compare against

    proceed = True
    if existing_row_count is not None and existing_row_count > 0:
        ratio = new_row_count / existing_row_count
        print(f"Existing row count: {existing_row_count} | New row count: {new_row_count} "
              f"({ratio:.0%} of existing)")
        if ratio < MIN_ROW_COUNT_RATIO and not FORCE_OVERWRITE:
            proceed = False
            print(f"\n❌ REFUSING TO OVERWRITE: new row count is only {ratio:.0%} of the "
                  f"existing table (threshold: {MIN_ROW_COUNT_RATIO:.0%}).")
            print("This looks like a partial/failed run, not a genuine data change.")
            print("If this drop is expected (e.g. wards were decommissioned), set")
            print("FORCE_OVERWRITE = True in the config cell and re-run.")
            write_skipped = True
            skip_reason = (f"new row count ({new_row_count}) was only {ratio:.0%} of the "
                           f"existing table ({existing_row_count}) - refused by the data-loss guard")

    if proceed:
        print(f"Writing {new_row_count} row(s) to [{TARGET_DATABASE}].[{TARGET_SCHEMA}].[{TARGET_TABLE}]...")
        try:
            df_parsed.to_sql(
                name=TARGET_TABLE,
                con=target_engine,
                schema=TARGET_SCHEMA,
                if_exists="replace",
                index=False,
            )
            print(f"✅ Load complete: [{TARGET_DATABASE}].[{TARGET_SCHEMA}].[{TARGET_TABLE}]")
        except Exception as e:
            print(f"\n❌ Failed to write to [{TARGET_DATABASE}].[{TARGET_SCHEMA}].[{TARGET_TABLE}]")
            print(f"  {str(e)[:400]}")
            print("  Likely cause: permissions (INSERT/CREATE TABLE), a column type")
            print("  mismatch, or the table being locked/in use by another session.")
            raise RuntimeError("Write to SQL Server failed - see above") from None

## 11. Verify the Load

Simple row-count check against the target table, confirming the write actually landed.

In [ ]:
if write_skipped:
    print("Skipping verification - no write was attempted this run.")
    print(f"Reason: {skip_reason}")
    print("Any row count in the target table right now is from a PREVIOUS")
    print("run, not this one.")
else:
    with target_engine.connect() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) FROM [{TARGET_SCHEMA}].[{TARGET_TABLE}]"))
        count = result.scalar()
    print(f"✓ Verified row count in target table: {count}")
    print(f"✅ DONE — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")